# 

## 0 · Imports & Base Paths

In [1]:
import pandas as pd
import os

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/BIOLOGICALPROCESS_CELLULARCOMP/ALL_BIOLOGICALPROCESS_CELLULARCOMP.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1 · Load Source Files

Each source DataFrame is loaded, columns lowercased, and `tail_id_is` / `head_id_is`  
set to the appropriate ID namespace (Quick_GO).

### 1.1 Monarch

In [2]:
Monarch_BP_CC = pd.read_csv(PROC_DIR + 'Monarch/Monarch_final/Human/BiologicalProcess_CellularComponent.csv')
Monarch_BP_CC.columns = Monarch_BP_CC.columns.str.lower()

Monarch_BP_CC['tail_id_is'] = 'Quick_GO'
Monarch_BP_CC['head_id_is'] = 'Quick_GO'
Monarch_BP_CC['relation'] = 'BiologicalProcess_CellularComponent'
Monarch_BP_CC['head_type'] = 'BiologicalProcess'
Monarch_BP_CC['tail_type'] = 'CellularComponent'

Monarch_BP_CC['kg_source'] = 'Monarch_KG'
Monarch_BP_CC['kg_type'] = 'Generalised'
Monarch_BP_CC['species'] = 'Homo sapiens'
print(f"Monarch BP→BP: {len(Monarch_BP_CC):,} rows")
Monarch_BP_CC.head(3)

Monarch BP→BP: 1,489 rows


,head,relation,tail,head_type,relation_type,tail_type,relation_source,head_detail_name,tail_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,species_species,species,kg_source,kg_type
0,GO:0060830,BiologicalProcess_CellularComponent,GO:0005929,BiologicalProcess,related_to,CellularComponent,infores:go,ciliary receptor clustering involved in smooth...,cilium,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens,Monarch_KG,Generalised
1,GO:0060988,BiologicalProcess_CellularComponent,GO:0060987,BiologicalProcess,related_to,CellularComponent,infores:go,lipid tube assembly,lipid tube,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens,Monarch_KG,Generalised
2,GO:0060997,BiologicalProcess_CellularComponent,GO:0043197,BiologicalProcess,related_to,CellularComponent,infores:go,dendritic spine morphogenesis,dendritic spine,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens,Monarch_KG,Generalised


## 2 · Consolidate into Unified Schema

All source DataFrames are aligned to `REQUIRED_COLS` (missing columns filled with `pd.NA`),  
then concatenated into `final_df`.

In [3]:
source_dfs = [
    Monarch_BP_CC,
    
]

aligned = []
for df in source_dfs:
    df = df.copy()
    # Drop any duplicated columns that might have resulted from original CSVs
    df = df.loc[:, ~df.columns.duplicated()]
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
final_df = final_df.dropna(subset=['head_detail_name', 'tail_detail_name']).reset_index(drop=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

Consolidated rows: 1,489


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,GO:0060830,BiologicalProcess_CellularComponent,GO:0005929,BiologicalProcess,related_to,CellularComponent,Monarch_KG,Quick_GO,Quick_GO,ciliary receptor clustering involved in smooth...,cilium,Generalised,Homo sapiens
1,GO:0060988,BiologicalProcess_CellularComponent,GO:0060987,BiologicalProcess,related_to,CellularComponent,Monarch_KG,Quick_GO,Quick_GO,lipid tube assembly,lipid tube,Generalised,Homo sapiens
2,GO:0060997,BiologicalProcess_CellularComponent,GO:0043197,BiologicalProcess,related_to,CellularComponent,Monarch_KG,Quick_GO,Quick_GO,dendritic spine morphogenesis,dendritic spine,Generalised,Homo sapiens
3,GO:0061024,BiologicalProcess_CellularComponent,GO:0016020,BiologicalProcess,related_to,CellularComponent,Monarch_KG,Quick_GO,Quick_GO,membrane organization,membrane,Generalised,Homo sapiens
4,GO:0061025,BiologicalProcess_CellularComponent,GO:0016020,BiologicalProcess,related_to,CellularComponent,Monarch_KG,Quick_GO,Quick_GO,membrane fusion,membrane,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1484,GO:0060121,BiologicalProcess_CellularComponent,GO:0032420,BiologicalProcess,related_to,CellularComponent,Monarch_KG,Quick_GO,Quick_GO,vestibular receptor cell stereocilium organiza...,stereocilium,Generalised,Homo sapiens
1485,GO:0060151,BiologicalProcess_CellularComponent,GO:0005777,BiologicalProcess,related_to,CellularComponent,Monarch_KG,Quick_GO,Quick_GO,peroxisome localization,peroxisome,Generalised,Homo sapiens
1486,GO:0060155,BiologicalProcess_CellularComponent,GO:0042827,BiologicalProcess,related_to,CellularComponent,Monarch_KG,Quick_GO,Quick_GO,platelet dense granule organization,platelet dense granule,Generalised,Homo sapiens
1487,GO:0060271,BiologicalProcess_CellularComponent,GO:0005929,BiologicalProcess,related_to,CellularComponent,Monarch_KG,Quick_GO,Quick_GO,cilium assembly,cilium,Generalised,Homo sapiens


## 3 · Sanity Check — Distinct Values

In [4]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'BiologicalProcess_CellularComponent'}
head_type           : {'BiologicalProcess'}
tail_type           : {'CellularComponent'}
relation_type       : {'related_to'}
kg_source           : {'Monarch_KG'}
head_id_is          : {'Quick_GO'}
tail_id_is          : {'Quick_GO'}


## 4 · NaN Audit (pre-dedup)

In [5]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 5 · Deduplication

Group on `(head, relation, tail)`. For `kg_source`, merge all unique sources with `::` delimiter.  
All other metadata columns take the first non-null value.

In [6]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': 'first',
    'species': 'first'
    
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 1,489  |  After dedup: 1,489


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,GO:0000001,BiologicalProcess_CellularComponent,GO:0005739,BiologicalProcess,related_to,CellularComponent,Monarch_KG,Quick_GO,Quick_GO,mitochondrion inheritance,mitochondrion,Generalised,Homo sapiens
1,GO:0000011,BiologicalProcess_CellularComponent,GO:0005773,BiologicalProcess,related_to,CellularComponent,Monarch_KG,Quick_GO,Quick_GO,vacuole inheritance,vacuole,Generalised,Homo sapiens
2,GO:0000027,BiologicalProcess_CellularComponent,GO:0015934,BiologicalProcess,related_to,CellularComponent,Monarch_KG,Quick_GO,Quick_GO,ribosomal large subunit assembly,large ribosomal subunit,Generalised,Homo sapiens


## 6 · Post-dedup NaN Audit & Source Distribution

In [7]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

display(pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
}))

print("\nkg_source values present:", set(final_df_dedup['kg_source']), set(final_df_dedup['kg_type']))

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0



kg_source values present: {'Monarch_KG'} {'Generalised'}


In [8]:
final_df_dedup['head_species'] = 'Homo sapiens'
final_df_dedup['tail_species'] = 'Homo sapiens'


## 7 · Save Output

In [9]:
import os
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 1,489 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/BIOLOGICALPROCESS_CELLULARCOMP/ALL_BIOLOGICALPROCESS_CELLULARCOMP.csv
